# Build an AI Agent with Grounding Tool

Welcome to week 3 of April's Developer Challenge on AI! This week you will combine what you have learned in week 1 and 2 and build an AI agent that uses the grounding service as a tool.

In [7]:
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

AICORE_ORCHESTRATION_DEPLOYMENT_URL = "https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/D276250e074fc028/v2"

In [ ]:
import json

from gen_ai_hub.document_grounding.client import RetrievalAPIClient
from gen_ai_hub.document_grounding.models.retrieval import (
    RetrievalSearchInput,
    RetrievalSearchFilter,
)
from gen_ai_hub.orchestration.models.document_grounding import DataRepositoryType

from crewai import Agent, Crew, Task
from crewai.tools import tool

In [ ]:
@tool("call_grounding_service")
def call_grounding_service(user_question: str) -> str:
    """Function to call the grounding service and retrieve relevant information based on the user's question."""
    retrieval_client = RetrievalAPIClient()

    search_filter = RetrievalSearchFilter(
        id="vector",
        dataRepositoryType=DataRepositoryType.VECTOR.value,
        dataRepositories=["9f9dd8b9-decb-441c-a395-7334416c0280"],
        searchConfiguration={
            "maxChunkCount": 2
        },
    )

    search_input = RetrievalSearchInput(
        query=user_question,
        filters=[search_filter],
    )

    response = retrieval_client.search(search_input)

    response_dict = json.dumps(response.model_dump(), indent=2)
    return response_dict

In [ ]:
# Create a Hamburg Social Welfare Case Manager Agent
welfare_agent = Agent(
    role="Hamburg Social Welfare Case Manager",
    goal="Process and manage social welfare applications for the city of Hamburg, matching citizens with appropriate social services and benefits based on their needs and circumstances. Use the call_grounding_service tool to retrieve relevant information from Hamburg's social services knowledge base to inform your assessments and recommendations.",
    backstory="You are an experienced social welfare case manager working for the city of Hamburg. You have deep knowledge of Hamburg's social services programs, eligibility requirements, local resources, and citizen support initiatives. Your role is to help vulnerable populations access the social welfare benefits they need.",
    llm="sap/gpt-4o",  # provider/llm - Using one of the models from SAP's model library in Generative AI Hub
    tools=[call_grounding_service],
    verbose=True
)

# Get user input
print("Hamburg Social Welfare Case Management System")
print("=" * 50)
user_question = input("\nPlease describe your social welfare inquiry or situation:\n> ")

# Create a task for the welfare case manager with user input
process_welfare_task = Task(
    description=f"Process the following social welfare inquiry for a Hamburg citizen: {user_question}\n\nBased on this inquiry and the information from the grounding service, determine eligibility for social welfare programs, recommend appropriate services, and create a personalized support plan.",
    expected_output="Structured social welfare assessment with eligibility determination, service recommendations, and personalized support plan for Hamburg citizens.",
    agent=welfare_agent
)

# Create a crew with the welfare agent
crew = Crew(
    agents=[welfare_agent],
    tasks=[process_welfare_task],
    verbose=True
)

# Execute the crew
if __name__ == "__main__":
    result = crew.kickoff()
    print("\n" + "="*50)
    print("Hamburg Social Welfare Assessment Report:")
    print("="*50)
    print(result)